In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
# go more complex
a =2.0
b = -3.0
c = 10.0
d = a *b +c
print(d)

In [ ]:
class Value:

    def __init__(self, data, _children = (), _op= '', label = ''):
        self.data = data
        self._prev = set(_children)
        self._op = _op
        self.label = label
        self.grad = 0
        self._backward = lambda: None 

    def __repr__(self):
        return f"Value(data={self.data}, op={self._op})" 
    
    
    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), '+')
        def _backward():
            self.grad += out.grad * 1.0
            other.grad += out.grad * 1.0
        out._backward = _backward
        return out
    
    def __radd__(self, other):
        return self + other
    
    
    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), '*')
        def _backward():
            self.grad += out.grad * other.data
            other.grad += out.grad * self.data
        out._backward = _backward
        return out
    
    def __rmul__(self, other):
        return self * other

    def tanh(self):
        n = self.data
        t = (math.exp(2*n) - 1 ) / (math.exp(2*n) + 1)
        out = Value(t, (self, ), 'tanh')
        def _backward():
            self.grad += out.grad * (1- t**2)
        out._backward = _backward

        return out
    
    def exp(self):
        t = math.exp(self.data) 
        out = Value(t, (self, ), 'exp')
        def _backward():
            self.grad += out.grad * t
        out._backward = _backward

        return out
    
    def __pow__(self, other):
        assert isinstance(other, (int, float))
        out =  Value(self.data**(other), (self, ), f'**{other}')
        def _backward():
            self.grad += out.grad * other * self.data ** (other-1)
        out._backward = _backward

        return out
    
    def __truediv__(self, other):
        return self * (other**-1)
    
    def neg(self):
        return self * -1
    
    def __sub__(self, other):
        return self + (-other)
    
    def __rsub__(self, other):
        return self + other
    


    def backward(self):
        # topological sort
        topo = []
        visited = set()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v) # CANNOT REVERSE WITH PUTTING BEFORE
        build_topo(self)
        self.grad = 1
        for node in topo[::-1]:
            node._backward()
x1 = Value(2.0, label = 'x1')
print(2*x1)
print(x1*2)

print(2+x1)
print(x1+2)

print(x1-1)
print(1-x1)

print(x1**2)

In [ ]:
a = Value(2)
b= Value(4)
a/b

In [ ]:
b = a

In [ ]:
b.grad =1
b.backward()

In [ ]:
x1.grad

In [ ]:
x2 = x1**3
x2.grad =1
x2.backward()

### Need to install graphviz
- pip install graphviz
- sudo apt update
- sudo apt install graphviz -y

Check by running:
- dot -V


In [ ]:
from graphviz import Digraph

def trace(root):
  # builds a set of all nodes and edges in a graph
  nodes, edges = set(), set()
  def build(v):
    if v not in nodes:
      nodes.add(v)
      for child in v._prev:
        edges.add((child, v))
        build(child)
  build(root)
  return nodes, edges

def draw_dot(root):
  dot = Digraph(format='svg', graph_attr={'rankdir': 'LR'}) # LR = left to right
  
  nodes, edges = trace(root)
  for n in nodes:
    uid = str(id(n))
    # for any value in the graph, create a rectangular ('record') node for it
    dot.node(name = uid, label = "{%s | data %.4f | grad %.4f }" % (n.label, n.data, n.grad), shape='record')
    if n._op:
      # if this value is a result of some operation, create an op node for it
      dot.node(name = uid + n._op, label = n._op)
      # and connect this node to it
      dot.edge(uid + n._op, uid)

  for n1, n2 in edges:
    # connect n1 to the op node of n2
    dot.edge(str(id(n1)), str(id(n2)) + n2._op)

  return dot


# def draw_dot(root):
#   dot = Digraph(format='svg', graph_attr={'rankdir': 'LR'}) # LR = left to right
  
#   nodes, edges = trace(root)
#   for n in nodes:
#     uid = str(id(n))
#     # for any value in the graph, create a rectangular ('record') node for it
#     dot.node(name = uid, label = "{%s | data %.4f | grad %.4f }" % (n.label, n.data, n.grad), shape='record')
#     # if n._op:
#       # if this value is a result of some operation, create an op node for it
#       # dot.node(name = uid + n._op, label = n._op)
#       # and connect this node to it
#       # dot.edge(uid + n._op, uid)

#   for n1, n2 in edges:
#     # connect n1 to the op node of n2
#     dot.edge(str(id(n1)), str(id(n2)) )

#   return dot

In [ ]:
x1 = Value(2.0, label = 'x1')
x2 = Value(0.0, label = 'x2')
w1 = Value(-3.0, label = 'w1')
w2 = Value(1.0, label = 'w2')
b = Value(6.8813735870195432, label = 'b')

x1w1 = x1*w1; x1w1.label = 'x1*w1'
x2w2 = x2*w2; x2w2.label = 'x2*w2'
x1w1x2w2 = x1w1 + x2w2; x1w1x2w2.label = 'x1*w1 + x2*w2'
n= x1w1x2w2 + b; n.label = 'n'
o = n.tanh()
o.backward()
draw_dot(o)

In [ ]:
x1 = Value(2.0, label = 'x1')
x2 = Value(0.0, label = 'x2')
w1 = Value(-3.0, label = 'w1')
w2 = Value(1.0, label = 'w2')
b = Value(6.8813735870195432, label = 'b')

x1w1 = x1*w1; x1w1.label = 'x1*w1'
x2w2 = x2*w2; x2w2.label = 'x2*w2'
x1w1x2w2 = x1w1 + x2w2; x1w1x2w2.label = 'x1*w1 + x2*w2'
n= x1w1x2w2 + b; n.label = 'n'
e = (2 * n).exp()
o = (e - 1 )/ (e+1)
o.label = 'o'
o.backward()

In [ ]:

draw_dot(o)

In [ ]:
# Using Pytorch

In [ ]:
import pytorch


x1 = torch.Tensor([2.0]).double();              x1.requires_grad = True
x2 = torch.Tensor([2.0]).double();              x2.requires_grad = True
w1 = torch.Tensor([2.0]).double();              w1.requires_grad = True
w2 = torch.Tensor([2.0]).double();              w2.requires_grad = True

b = torch.Tensor([6.8813735870195432]).double(); b.requires_grad = True

n= x1*w1 + x2*w2 + b
o = torch.tanh(n)
print(o.data.item())

print('------')
print('x1', x1.grad.item())
print('x2', x2.grad.item())
print('w1', w1.grad.item())
print('w2', w2.grad.item())

In [ ]:
# NN

In [ ]:
class Neuron:
    def __init__(self, nin):
        self.w = [Value(random.uniform(-1, 1)) for _ in range(nin)]
        self.b  Value(random.uniform(-1, 1))

    def __call__(self, x):
        act = sum([xi*wi for xi, wi in zip(x, self.w)],  self.b)
        out = act.tanh()
        return out

x = [2, 4]
n = Neuron(2)
n(x)

In [ ]:
class Layer:
    def __init__(self, nin, nout):
        self.neurons = [Neuron(nin) for  _ in range(nout)]

    def __call__(self, x):
        outs= [n(x) for n in self.neurons]
        return outs[0] if len(outs) == 1 else outs

In [ ]:
class MLP:
    def __init__(nin, nouts):#nouts listdefininig the size of the layers
        sz = [nin] + nouts
        self.layers = [Layer(sz[i], sz[i+1]) for i in range(len(sz)-1)]

    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

In [ ]:
n = MLP(3, [4, 4, 1])

In [ ]:
xs = [
    [2, 3, -1],
    [3, -1, 0],
    [0.5, 1, -1],
    [1, 1, -1]
]
ys = [1, -1, -1, 1]
ypred = [n(x) for x in xs]
ypred

In [ ]:
loss = sum([ (yout - ygrt)**2 for yout, ygrt in  zip(ypred, ys)])